[⬅ Back to the examples](https://s-celles.github.io/anywidget-instruments/try/)

# Signal analysis

A 1 s record sampled at 2 kHz. **Run all the cells**, then choose the waveform,
its frequency and the noise level: the signal, its spectrum, the RMS level and
the dominant frequency are recomputed by the control callbacks.

**References:** SI prefixes (engineering units). These standards inspired
the widgets; the library does not claim conformity with them ([Standards and
references](https://s-celles.github.io/anywidget-instruments/standards/)).

> **The code is hidden.** To see a cell's code, click the `⋯` bar above its result; to see all of it, choose **View ▸ Expand All Code**.

In [ ]:
# JupyterLite: install the package (bundled with this site) in the browser kernel
%pip install -q anywidget-instruments

In [ ]:
import numpy as np

import anywidget_instruments as ai

FS, N = 2000.0, 2000
shape = ai.SelectorSwitch("SINE", positions=["SINE", "SQUARE", "TRIANGLE"], label="Waveform")
freq = ai.Knob(50, min=1, max=200, step=1, unit="Hz", label="Frequency", size=(130, 130))
noise = ai.Knob(0.1, min=0, max=1, step=0.01, label="Noise", size=(130, 130))
signal = ai.WaveformChart(
    history=200,
    dt=1 / FS,
    x_unit="s",
    unit="V",
    y_min=-3,
    y_max=3,
    traces=[{"name": "x(t)"}],
    label="Signal (first 100 ms)",
    size=(520, 200),
)
spectrum = ai.WaveformChart(
    history=N // 2 + 1,
    dt=FS / N,
    x_unit="Hz",
    y_min=0,
    y_max=1.2,
    traces=[{"name": "|X(f)|"}],
    label="Spectrum",
    size=(520, 200),
)
rms = ai.VUMeter(max=1.5, format="%.2f", unit="V", label="RMS", size=(60, 180))
peak = ai.SevenSegment(0, digits=5, decimals=0, label="Dominant frequency (Hz)")


def update(_change=None):
    t = np.arange(N) / FS
    phase = 2 * np.pi * freq.value * t
    wave = {
        "SINE": np.sin(phase),
        "SQUARE": np.sign(np.sin(phase)),
        "TRIANGLE": 2 / np.pi * np.arcsin(np.sin(phase)),
    }[shape.value]
    x = wave + noise.value * np.random.default_rng(0).standard_normal(N)
    mag = np.abs(np.fft.rfft(x * np.hanning(N))) / (N / 4)
    signal.clear()
    signal.append(x[:200])
    spectrum.clear()
    spectrum.append(mag)
    with ai.batch():
        rms.value = float(np.sqrt(np.mean(x**2)))
        peak.value = float(np.fft.rfftfreq(N, 1 / FS)[np.argmax(mag[1:]) + 1])


for control in (shape, freq, noise):
    control.on_change(update)
update()

ai.Panel([shape, freq, noise, rms, signal, spectrum, peak], columns=4)

In [ ]:
ai.theme_switch()  # light / system / dark theme (STYLE-008)